In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train_df = pd.read_parquet('train.parquet')
test_df = pd.read_parquet('test.parquet')

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Attack
0,80,5169956,8,6,1101,4222,410,0,137.625,185.758628,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DoS
1,80,229,2,0,12,0,6,6,6.000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DoS
2,80,5001928,3,1,12,0,6,0,4.000,3.464102,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DoS
3,80,5125872,8,7,1659,2514,467,0,207.375,221.278065,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DoS
4,80,214,2,0,12,0,6,6,6.000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DoS


In [3]:
from utils import handle_values
train_df = handle_values(train_df.copy())
test_df = handle_values(test_df.copy())

In [4]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df['Attack']= encoder.fit_transform(train_df['Attack'])
train_df['Attack'].value_counts()

Attack
0    1720966
4     189135
3      29999
6      25501
2       7417
7       1718
1        974
5         23
Name: count, dtype: int64

In [ ]:
import joblib
joblib.dump(encoder,r"scalers\label_enoder.joblib")


Attack
0    395106
3     98022
6     79290
4      4871
2      2039
1       981
7       433
5        24
Name: count, dtype: int64

In [6]:
test_df['Attack']= encoder.transform(test_df['Attack'])
test_df['Attack'].value_counts()

Attack
0    395106
3     98022
6     79290
4      4871
2      2039
1       981
7       433
5        24
Name: count, dtype: int64

In [5]:
# Printing corresponding attack type for each encoded value
encoded_values = train_df['Attack'].unique()
for val in sorted(encoded_values):
    print(f"{val}: {encoder.inverse_transform([val])[0]}")

0: BENIGN
1: Bot
2: Brute Force
3: DDoS
4: DoS
5: Other Attacks
6: Port Scan
7: Web Attack


In [11]:
print(f"Train Correlation Analysis:")
corr_train = train_df.corr(numeric_only=True).round(3)

Train Correlation Analysis:


In [12]:
print(f"Test Correlation Analysis:")
corr_test = test_df.corr(numeric_only=True).round(3)

Test Correlation Analysis:


Destination Port              -0.144
Flow Duration                  0.391
Total Fwd Packets             -0.002
Total Backward Packets        -0.003
Total Length of Fwd Packets   -0.007
                               ...  
Idle Mean                      0.600
Idle Std                       0.108
Idle Max                       0.603
Idle Min                       0.594
Attack                         1.000
Name: Attack, Length: 79, dtype: float64


In [ ]:
attack_corr = corr_train['Attack']
THRESHOLD = 0.1
pos_corr = attack_corr[(attack_corr > THRESHOLD) & (attack_corr < 1)]
pos_corr = pos_corr.sort_values(ascending=False)
print("Features with positive correlation with 'Attack' in Descending Order:\n")
for i, (feature, corr_value) in enumerate(pos_corr.items(), start=1):
    print(f"{i:<3} {feature:<24} : {corr_value}")

Features with positive correlation with 'Attack' in Descending Order:

1   Bwd Packet Length Std    : 0.642
2   Fwd IAT Std              : 0.633
3   Bwd Packet Length Max    : 0.628
4   Packet Length Std        : 0.626
5   Avg Bwd Segment Size     : 0.624
6   Bwd Packet Length Mean   : 0.624
7   Max Packet Length        : 0.609
8   Idle Max                 : 0.603
9   Idle Mean                : 0.6
10  Fwd IAT Max              : 0.599
11  Flow IAT Max             : 0.598
12  Idle Min                 : 0.594
13  Packet Length Variance   : 0.593
14  Average Packet Size      : 0.558
15  Packet Length Mean       : 0.557
16  Flow IAT Std             : 0.521
17  Fwd IAT Total            : 0.392
18  Flow Duration            : 0.391
19  FIN Flag Count           : 0.359
20  Bwd IAT Std              : 0.294
21  Flow IAT Mean            : 0.289
22  Fwd IAT Mean             : 0.269
23  Bwd IAT Max              : 0.243
24  ACK Flag Count           : 0.116
25  Idle Std                 : 0.108


In [33]:
print(f"Imp features = {len(pos_corr)}")

Imp features = 25


In [38]:
type(pos_corr)

pandas.core.series.Series

In [ ]:
imp_features = pos_corr.index.tolist()

['Bwd Packet Length Std',
 'Fwd IAT Std',
 'Bwd Packet Length Max',
 'Packet Length Std',
 'Avg Bwd Segment Size',
 'Bwd Packet Length Mean',
 'Max Packet Length',
 'Idle Max',
 'Idle Mean',
 'Fwd IAT Max',
 'Flow IAT Max',
 'Idle Min',
 'Packet Length Variance',
 'Average Packet Size',
 'Packet Length Mean',
 'Flow IAT Std',
 'Fwd IAT Total',
 'Flow Duration',
 'FIN Flag Count',
 'Bwd IAT Std',
 'Flow IAT Mean',
 'Fwd IAT Mean',
 'Bwd IAT Max',
 'ACK Flag Count',
 'Idle Std']

In [34]:
std = train_df.std(numeric_only = True)
zero_std_cols = std[std == 0].index.tolist()
zero_std_cols #DEAD or Useless Featuers

['Bwd PSH Flags',
 'Bwd URG Flags',
 'Fwd Avg Bytes/Bulk',
 'Fwd Avg Packets/Bulk',
 'Fwd Avg Bulk Rate',
 'Bwd Avg Bytes/Bulk',
 'Bwd Avg Packets/Bulk',
 'Bwd Avg Bulk Rate']

In [40]:
train_df.to_parquet('train_final.parquet')
test_df.to_parquet('test_final.parquet')